# Data Preparation Pipeline

**Objective:** Prepare the engineered ICU dataset for machine learning by:

- preventing patient leakage,
- creating train/test splits,
- imputing missing values,
- scaling numeric variables,
- encoding categorical variables,
- exporting ML-ready datasets and preprocessing artifacts.

### Table of Contents
  
1. [Configuration and Imports](#Configuration-and-Imports)  
2. [Load Engineered Features](#Load-Engineered-Features)  
3. [Initial Data Audit](#Initial-Data-Audit)  
4. [Feature Type Detection](#Feature-Type-Detection)  
5. [Grouped Train/Test Split](#Grouped-TrainTest-Split)  
6. [Preprocessing Strategy](#Preprocessing-Strategy)  
7. [Build Preparation Pipeline](#Build-Preparation-Pipeline)  
8. [Fit and Transform Data](#Fit-and-Transform-Data)  
9. [Export Prepared Datasets](#Export-Prepared-Datasets)  
10. [Save Preparation Artifacts](#Save-Preparation-Artifacts)  
11. [Final Remarks](#Final-Remarks)

### Configuration and Imports
[Back to Table of Contents](#Table-of-Contents)

In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

print(ROOT_DIR)

c:\Users\catar\OneDrive\Ambiente de Trabalho\fcup\3 ano\2\CDLE\ICU-LengthOfStay-Prediction


In [2]:
import pandas as pd
from pathlib import Path
import joblib
import logging;

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

ID_COLS = ["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID"]
TARGET_COL = "LOS"

ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models"

FEATURE_SET = "9tables"
WINDOW_SIZE = "24h"
EXPERIMENT_NAME = f"{FEATURE_SET}_{WINDOW_SIZE}"

EXPERIMENT_DATA_DIR = DATA_DIR / EXPERIMENT_NAME
EXPERIMENT_MODELS_DIR = MODELS_DIR / EXPERIMENT_NAME
EXPERIMENT_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = EXPERIMENT_DATA_DIR / f"features.csv"
TRAIN_PATH = EXPERIMENT_DATA_DIR / f"train.csv"
TEST_PATH = EXPERIMENT_DATA_DIR / f"test.csv"
PREPROCESSOR_PATH = EXPERIMENT_MODELS_DIR / f"preprocessor.joblib"

### Load Engineered Features

The preparation stage starts from the engineered ICU feature table generated during preprocessing.

Each row represents a single ICU stay (`ICUSTAY_ID`).

[Back to Table of Contents](#Table-of-Contents)

In [14]:
features = pd.read_csv(INPUT_PATH)

print("Dataset shape:", features.shape)

display(features.head(2))

Dataset shape: (59125, 588)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,DBSOURCE,FIRST_CAREUNIT,LAST_CAREUNIT,LOS,GENDER,AGE,ADMISSION_TYPE,...,drug_bag_count,drug_hydromorphone_dilaudid__count,drug_acetaminophen_count,drug_vial_count,drug_furosemide_count,drug_metoprolol_count,drug_potassium_chloride_count,drug_heparin_count,drug_pantoprazole_count,drug_aspirin_count
0,42820,127889,244306,METAVISION,MICU,MICU,0.6651,F,76,EMERGENCY,...,1.0,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN
1,94079,169529,294608,METAVISION,TSICU,TSICU,5.0282,M,78,EMERGENCY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


### Initial Dataset Audit

Before splitting and transforming the dataset, it is important to validate:

- dataset dimensions,
- identifier availability,
- target availability,
- missing-value patterns,
- approximate feature composition.

[Back to Table of Contents](#Table-of-Contents)

In [15]:
print("Rows:", len(features))
print("Columns:", features.shape[1])

for col in ID_COLS + [TARGET_COL]:
    print(f"{col}: {col in features.columns}")

missing_ratio = (
    features.isna()
    .mean()
    .sort_values(ascending=False)
)

display(missing_ratio.head(20).rename("missing_ratio").to_frame())

Rows: 59125
Columns: 588
SUBJECT_ID: True
HADM_ID: True
ICUSTAY_ID: True
LOS: True


,missing_ratio
output_41707_last,0.996262
output_41707_count,0.996262
output_41707_sum,0.996262
output_226589_sum,0.993302
output_226589_count,0.993302
output_226589_last,0.993302
output_40048_sum,0.988651
output_40048_count,0.988651
output_40048_last,0.988651
output_40049_last,0.988567


### Feature Type Detection

The preprocessing pipeline separates features into:

- numeric variables,
- categorical variables.

This distinction is required because each feature type uses a different preprocessing strategy.

[Back to Table of Contents](#Table-of-Contents)

In [16]:
feature_cols = [
    col for col in features.columns
    if col not in ID_COLS + [TARGET_COL]
]

categorical_cols = (
    features[feature_cols]
    .select_dtypes(include=["object", "string", "category"])
    .columns
    .tolist()
)

numeric_cols = [
    col for col in feature_cols
    if col not in categorical_cols
]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 573
Categorical columns: 11


### Grouped Train/Test Split

The dataset is split using `GroupShuffleSplit` with `SUBJECT_ID` as the grouping variable.

This prevents patient leakage by ensuring that the same patient cannot appear in both training and test datasets.

[Back to Table of Contents](#Table-of-Contents)

In [17]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(features, groups=features["SUBJECT_ID"])
)

train = features.iloc[train_idx].copy()
test = features.iloc[test_idx].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (47333, 588)
Test shape: (11792, 588)


### Leakage Validation

The overlap between train and test patients is verified to confirm that grouped splitting worked correctly.

[Back to Table of Contents](#Table-of-Contents)

In [18]:
train_subjects = set(train["SUBJECT_ID"].unique())
test_subjects = set(test["SUBJECT_ID"].unique())

overlap = train_subjects.intersection(test_subjects)

print("Train subjects:", len(train_subjects))
print("Test subjects:", len(test_subjects))
print("Overlap:", len(overlap))

Train subjects: 35992
Test subjects: 8998
Overlap: 0


### Preprocessing Strategy

+ #### Numeric Features

    - median imputation,
    - missing indicators,
    - standard scaling.

+ #### Categorical Features

    - most frequent imputation,
    - one-hot encoding.

The preprocessing pipeline is fitted only on the training set to prevent data leakage.

[Back to Table of Contents](#Table-of-Contents)

### Build Preparation Pipeline
[Back to Table of Contents](#Table-of-Contents)

In [19]:
transformers = []

if numeric_cols:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )

    transformers.append(("num", numeric_pipeline, numeric_cols))

if categorical_cols:
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )),
        ]
    )

    transformers.append(("cat", categorical_pipeline, categorical_cols))

preprocessor = ColumnTransformer(
    transformers=transformers
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### Fit and Transform Data

The preprocessing pipeline is fitted only on the training data.

The learned transformations are then applied to both train and test datasets.

[Back to Table of Contents](#Table-of-Contents)

In [20]:
train_values = preprocessor.fit_transform(
    train[feature_cols]
)

test_values = preprocessor.transform(
    test[feature_cols]
)

transformed_cols = [
    col.replace("num__", "").replace("cat__", "")
    for col in preprocessor.get_feature_names_out()
]

train_out = pd.concat(
    [
        train[ID_COLS + [TARGET_COL]].reset_index(drop=True),
        pd.DataFrame(train_values, columns=transformed_cols),
    ],
    axis=1,
)

test_out = pd.concat(
    [
        test[ID_COLS + [TARGET_COL]].reset_index(drop=True),
        pd.DataFrame(test_values, columns=transformed_cols),
    ],
    axis=1,
)

print("Prepared train shape:", train_out.shape)
print("Prepared test shape:", test_out.shape)

Prepared train shape: (47333, 1320)
Prepared test shape: (11792, 1320)


### Export Prepared Datasets
[Back to Table of Contents](#Table-of-Contents)

In [21]:
train_out.to_csv(TRAIN_PATH, index=False)
test_out.to_csv(TEST_PATH, index=False)

print("Saved:", TRAIN_PATH)
print("Saved:", TEST_PATH)

Saved: ..\data\processed\9tables_24h\train.csv
Saved: ..\data\processed\9tables_24h\test.csv


## Save Preparation Artifacts

The fitted preprocessing pipeline is saved to support:

- reproducibility,
- consistent future inference,
- deployment compatibility.

[Back to Table of Contents](#Table-of-Contents)

In [22]:
joblib.dump(preprocessor, PREPROCESSOR_PATH)

print("Saved:", PREPROCESSOR_PATH)

Saved: ..\data\processed\9tables_24h\preprocessor.joblib


## Final Remarks

The preparation stage produced:

- leakage-safe train/test datasets,
- fully numeric ML-ready feature matrices,
- a reusable preprocessing pipeline artifact.

These outputs are now ready for model training, benchmarking, tuning, and evaluation.

[Back to Table of Contents](#Table-of-Contents)